# X03 · GPU execution model and systolic arrays

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/10_compute_deployment/X03_gpu_execution_model_and_systolic_arrays.ipynb)

**Track:** Compute and deployment · **Skill axis (sheet):** Compute and deployment · **Time:** about 60 min

**Skill Hub concepts:** Warp, block, coalescing; Systolic array

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("X03")

## Map: how parallel hardware actually runs your math

**GPU (CUDA terms):** a kernel launches a **grid** of **blocks**; each block has up to 1024 **threads**; the hardware
runs threads in **warps** of 32 that execute the same instruction (single instruction, multiple threads).

| Concept | Rule of thumb | Cost of ignoring it |
|---|---|---|
| Blocks for $n$ elements | $\lceil n / \text{threads} \rceil$ or a grid-stride loop | missed elements or out-of-bounds writes |
| Warp divergence | keep branches uniform within 32 threads | both branch paths run serially |
| **Coalescing** | thread $i$ of a warp touches address $base + 4i$ | each 128-byte memory transaction serves fewer threads |
| Shared memory | reuse tiles loaded once per block | repeated global memory traffic |

**Systolic array (TPU, many neural processing units):** a grid of multiply-accumulate processing elements (PEs). In an
**output-stationary** $M \times N$ array, PE $(i, j)$ keeps $C_{ij}$; row $i$ of $A$ enters from the left delayed by $i$
cycles, column $j$ of $B$ enters from the top delayed by $j$ cycles; values shift one PE right (for $A$) or down (for
$B$) per cycle. PE $(i, j)$ performs its $k$-th multiply at cycle $k + i + j$ (counting from 0), so the whole
product takes
$$\text{cycles} = K + M + N - 2$$
Each operand loaded once from memory is reused across a whole row or column: very high arithmetic intensity.

In [ ]:
import math, os, shutil
import numpy as np
if shutil.which("nvidia-smi") is None:
    os.environ.setdefault("NUMBA_ENABLE_CUDASIM", "1")      # no GPU: run kernels in Numba's CUDA simulator
from numba import cuda
print("CUDA simulator" if os.environ.get("NUMBA_ENABLE_CUDASIM") == "1" else "real GPU")

## Task 1: launch geometry

In [ ]:
def blocks_for(n, threads):
    raise NotImplementedError  # TODO

def warps_per_block(threads):
    raise NotImplementedError  # TODO

lab.check("launch", blocks_for, warps_per_block)

## Task 2: a grid-stride SAXPY kernel, $out = a x + y$
Use `cuda.grid(1)` for the global index and `cuda.gridsize(1)` for the stride so any launch size covers every element.

In [ ]:
@cuda.jit
def saxpy(a, x, y, out):
    pass  # TODO

lab.check("saxpy", saxpy)

## Task 3: coalescing
`segments_touched(addresses, segment=128)`: number of distinct aligned 128-byte segments a warp's addresses fall in.
`warp_addresses(base, stride_elements, elem_bytes=4)`: the 32 byte addresses when thread $t$ reads element $t \cdot stride$.

In [ ]:
def segments_touched(addresses, segment=128):
    raise NotImplementedError  # TODO

def warp_addresses(base, stride_elements, elem_bytes=4):
    raise NotImplementedError  # TODO

lab.check("coalescing", segments_touched, warp_addresses)

## Task 4: cycle-level output-stationary systolic array
Simulate cycle by cycle with registers `a_reg` and `b_reg` (shape M x N) and accumulators `acc`. Each cycle, first
shift `a_reg` right by one column and `b_reg` down by one row, then feed the edges: PE $(i, 0)$ receives
$A[i, c - i]$ when $0 \le c - i < K$ (else 0), PE $(0, j)$ receives $B[c - j, j]$ similarly, then every PE adds
`a_reg * b_reg` to `acc`. Return `(C, cycles)` where `cycles` counts until the last useful multiply.

In [ ]:
def systolic_matmul(A, B):
    raise NotImplementedError  # TODO

lab.check("systolic", systolic_matmul)

## Checkpoint: Math
**M1 (Apply).** Blocks of 256 threads for 1,000,000 elements (one element per thread)?

**M2 (Apply).** Warps in a 256-thread block?

**M3 (Apply).** Cycles for a 4 x 4 by 4 x 4 product on a 4 x 4 output-stationary array?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** In a kernel, even threads take branch A (20 instructions) and odd threads take branch B (30
instructions). Roughly how long does a warp spend?
- a) 30 instructions: the longer branch
- b) 50 instructions: both paths run one after the other with half the threads masked
- c) 25 instructions: the average

**K2 (Understand).** Explain why stride-1 access is fast and stride-32 access is slow on a GPU, using your Task 3
numbers, and one way to fix a kernel that needs column access of a row-major matrix.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `x03.h`).
3. In `roboprep/checks/x03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()